# GeoJSONをPythonで扱う
読み込み・作成・検証・地図表示。東京駅付近の**架空の説明用データ**だけを使います。実際の経路・施設境界ではありません。

[記事引き継ぎ](../docs/geojson-python-article-handoff.md)に環境・実測結果・公式出典をまとめています。既存uv環境で、作業ディレクトリを `projects/poi` にして順番に実行してください。

## 1. jsonで読む
FeatureCollectionの中にPoint・LineString・Polygonを1つずつ入れます。JSONの構文とGeoJSONの構造は別の確認です。

In [ ]:
import json
import geopandas as gpd
import pandas as pd
from IPython.display import display, SVG
from geoai_open_lab.geojson_tutorial import example_collection, example_frame, basic_check, run, OUT

data = json.loads(json.dumps(example_collection(), ensure_ascii=False))
print(data["type"], len(data["features"]))
[(f["geometry"]["type"], f["properties"]["name"]) for f in data["features"]]

## 2. GeoDataFrameで読む
1つのFeatureが1行、propertiesが属性列になります。ここでのCRS指定は、入力がWGS 84経緯度であると分かっているためです。座標を変換する操作ではありません。

In [ ]:
gdf = gpd.GeoDataFrame.from_features(data, crs="EPSG:4326")
print(gdf.crs)
display(gdf.assign(geometry_type=gdf.geom_type)[["name", "fictional", "geometry_type"]])

## 3. 作成・保存・再読込
example_frame()はShapelyのPoint・LineString・Polygonから作ります。Polygonは外周を反時計回りにします。to_wgs84=Trueだけで全GeoJSONの正しさを保証するわけではありません。

In [ ]:
created = example_frame()
OUT.mkdir(parents=True, exist_ok=True)
path = OUT / "tutorial_saved.geojson"
path.write_text(created.to_json(to_wgs84=True, drop_id=True, ensure_ascii=False, allow_nan=False), encoding="utf-8")
loaded = gpd.read_file(path, engine="pyogrio")
assert all(a.equals_exact(b, 0) for a, b in zip(created.geometry, loaded.geometry))
assert created.drop(columns="geometry").equals(loaded.drop(columns="geometry"))
print("3地物の形状・属性が一致")

## 4. 誤りを段階別に確かめる
json構文 → 基本構造 → 幾何 → 内容。簡易チェックはこの記事の2D・3種類だけが対象で、RFC 7946の完全な検証器ではありません。

In [ ]:
report = run()
rows = pd.DataFrame(report["cases"])
display(rows[["case", "json_ok", "from_features_ok", "read_file_ok", "shapely_valid", "basic_issues", "content_issues"]])

未閉鎖Polygonはfrom_featuresでは補完されましたが、今回のread_file経路ではエラー。経緯度が逆のPointでもis_validはTrue。自己交差は読めてもFalseです。読み込み方法とバージョンを明記しましょう。

## 5. 出力方法も確認する
投影座標のto_json既定値やto_file既定値では旧crsメンバーが出力される場合があります。今回のto_fileはpyogrioを明示しています。

In [ ]:
display(pd.DataFrame(report["exports"])[["method", "has_crs", "first_coordinates"]])
print(report["versions"])

## 6. 地図で位置と属性を見る
保存はWGS 84の経度・緯度、表示は東京向けEPSG:6677。外部タイルは使用しません。

In [ ]:
from geoai_open_lab.geojson_figures import render
render()
display(SVG(filename=str(OUT / "final_map.svg")))
display(SVG(filename=str(OUT / "validation_layers.svg")))

## まとめ
読み込めること、幾何が妥当なこと、座標や内容が正しいことは別です。GeoJSONには現実の正しさを保証する仕組みはありません。ここではMulti系、穴、日付変更線、3D、大容量ファイルは扱いません。距離や面積を緯度経度の度で計算しないようにします。